# Teste: vector store para CUIs

In [1]:
import pandas as pd 
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import sys
import os

# Get absolute path to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

In [2]:
print(f"Project root: {project_root}")

Project root: /home/ia368/projetos/imageclef2026-rag


# Import CUIs

In [3]:
dataset_path = "../../datasets/imageclef/dataset/cui_mapping.csv"

cui_df = pd.read_csv(dataset_path)
cui_df.head()

,CUI,Canonical name
0,C0016522,"Foramen Ovale, Patent"
1,C0457846,Cervical segment of spinal cord
2,C0578537,Cavitation of lung
3,C0179751,Epidural catheter
4,C3489393,Hiatal Hernia


In [10]:
len(cui_df)

1934

In [5]:
cui_2026 = pd.read_csv("../dev_caption/concepts.csv")
cui_2026.head()

,ID,CUIs
0,ImageCLEFmedical_Caption_2026_train_0,C0040405
1,ImageCLEFmedical_Caption_2026_train_1,C1306645;C0817096;C0442800;C0018787;C0242073
2,ImageCLEFmedical_Caption_2026_train_2,C0040405;C0856747
3,ImageCLEFmedical_Caption_2026_train_3,C0041618
4,ImageCLEFmedical_Caption_2026_train_4,C0040405;C0040053


In [6]:
# Tratar coluna CUIs (podem existir múltiplos valores separados por ';')
cui_unique_2026 = sorted({
    cui.strip()
    for cell in cui_2026["CUIs"].dropna().astype(str)
    for cui in cell.split(";")
    if cui.strip()
})

print(f"Total de CUIs únicos: {len(cui_unique_2026)}")
print(f"Exemplo (primeiros 10): {cui_unique_2026[:10]}")

Total de CUIs únicos: 2646
Exemplo (primeiros 10): ['C0000726', 'C0000741', 'C0000833', 'C0000846', 'C0000962', 'C0001074', 'C0001080', 'C0001162', 'C0001168', 'C0001208']


In [ ]:
# todos cui_unique_2026 estao em cui_df['CUI'] ?

if all(cui in cui_df['CUI'].values for cui in cui_unique_2026):
    print("Todos os CUIs únicos de 2026 estão presentes no DataFrame de mapeamento.")
else:
    missing_cuis = [cui for cui in cui_unique_2026 if cui not in cui_df['CUI'].values]
    print(f"Total de CUIs faltantes: {len(missing_cuis)}")
    print(f"CUIs faltantes: {missing_cuis}")

Total de CUIs faltantes: 777
CUIs faltantes: ['C0000741', 'C0001080', 'C0002726', 'C0002760', 'C0002871', 'C0003486', 'C0003490', 'C0003509', 'C0003707', 'C0004031', 'C0004626', 'C0005767', 'C0005898', 'C0005939', 'C0005967', 'C0006111', 'C0006142', 'C0006270', 'C0006382', 'C0006497', 'C0006826', 'C0007113', 'C0007130', 'C0007138', 'C0007194', 'C0007281', 'C0007282', 'C0007572', 'C0007787', 'C0007847', 'C0007873', 'C0008034', 'C0008520', 'C0008827', 'C0009201', 'C0009917', 'C0009995', 'C0010068', 'C0010097', 'C0010276', 'C0010417', 'C0010678', 'C0010964', 'C0011119', 'C0011302', 'C0011325', 'C0011443', 'C0011981', 'C0012634', 'C0014013', 'C0014038', 'C0014520', 'C0014533', 'C0014609', 'C0014850', 'C0014856', 'C0014858', 'C0014866', 'C0015029', 'C0015183', 'C0015388', 'C0015914', 'C0016045', 'C0016109', 'C0016540', 'C0017070', 'C0017145', 'C0017168', 'C0017181', 'C0017525', 'C0017589', 'C0018023', 'C0018220', 'C0018418', 'C0018427', 'C0018784', 'C0018826', 'C0018870', 'C0018934', 'C0018

# Gerar embeddings para CUIs

In [4]:
# Carregar utilitários do projeto
from utils.f_utils import load_config, load_models

In [5]:
# Carregar configuração e modelo MedSigLip
config_path = "../configs/IC2024_valid_emb.yaml"
config = load_config(config_path)

processor, model, device = load_models(config, device='cuda')
print(f"Using device: {device}")
print(f"Model loaded: {config['medsiglip']['model']}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Using device: cuda
Model loaded: google/medsiglip-448


In [6]:
# Dataset para processar apenas textos (Canonical names)
class TextOnlyDataset(Dataset):
    def __init__(self, texts, processor, max_length=64):
        """
        texts: lista de textos (Canonical names)
        processor: processor do MedSigLip
        max_length: tamanho máximo do texto
        """
        self.texts = texts
        self.processor = processor
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        
        # Processar apenas o texto (sem imagem)
        encoding = self.processor(
            text=text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # Remove batch dimension
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        return encoding

In [7]:
# Função para gerar embeddings de texto
def generate_text_embeddings(model, dataloader, device):
    """
    Gera embeddings apenas para textos usando o encoder de texto do MedSigLip
    """
    model.eval()
    model.to(device)
    
    all_text_embeds = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Generating text embeddings"):
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # Gerar embeddings de texto
            text_embeds = model.get_text_features(
                input_ids=batch["input_ids"]
            )
            
            # Normalização (para cosine similarity)
            text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
            
            all_text_embeds.append(text_embeds.cpu())
    
    all_text_embeds = torch.cat(all_text_embeds, dim=0)
    return all_text_embeds

In [8]:
# Preparar dados: extrair Canonical names
canonical_names = cui_df['Canonical name'].tolist()

print(f"Total de Canonical names: {len(canonical_names)}")
print(f"Exemplo: {canonical_names[0]}")

Total de Canonical names: 1934
Exemplo: Foramen Ovale, Patent


In [9]:
# Criar dataset e dataloader
text_dataset = TextOnlyDataset(
    texts=canonical_names,
    processor=processor,
    max_length=64
)

text_dataloader = DataLoader(
    text_dataset,
    batch_size=32,
    shuffle=False,
    pin_memory=True
)

print(f"Dataset criado com {len(text_dataset)} amostras")
print(f"Número de batches: {len(text_dataloader)}")

Dataset criado com 1934 amostras
Número de batches: 61


In [ ]:
# Gerar embeddings
cui_text_embeddings = generate_text_embeddings(
    model=model,
    dataloader=text_dataloader,
    device=device
)

print(f"\n✅ Embeddings gerados com sucesso!")
print(f"📊 Shape dos embeddings: {cui_text_embeddings.shape}")
print(f"📊 Tipo: {type(cui_text_embeddings)}")

In [ ]:
# Salvar embeddings (opcional)
output_dir = "../artifacts/embeddings"
os.makedirs(output_dir, exist_ok=True)

torch.save(cui_text_embeddings, os.path.join(output_dir, "cui_canonical_names_embeddings.pt"))
torch.save(canonical_names, os.path.join(output_dir, "cui_canonical_names.pt"))

print(f"✅ Embeddings salvos em {output_dir}")
print(f"📄 Arquivo de embeddings: cui_canonical_names_embeddings.pt")
print(f"📄 Arquivo de nomes: cui_canonical_names.pt")